<a href="https://colab.research.google.com/github/ngophuctaivp20-crypto/Final-Project-/blob/main/Saving_Goal_Coach_Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

6 Steps Process:

1.understand the problem

Many people they want to save their money to buy things such as laptop, watch, or something valuable.But they don't know how to create a good saving plan. Thus, I will introduce the Saving Goal Coach that let the user set goals with deadlines, track progress,
analyse spending from a CSV of transactions, and ask an AI coach for advice based on their real numbers. It gives general budgeting guidance, not financial
advice.

2.identify the inputs and outputs

The inputs should be a savings goal, which consists of a goal name, a target amount, a deadline in YYYY-MM-DD format and any savings the user already holds. Therefore, I want the system produces output like a summary showing the amount saved, the percentage complete, the number of months remaining, the monthly contribution required to meet the deadline and an overall progress status.

Other should be transaction data, supplied as a CSV file containing the date, description, category and amount of each transaction, with income recorded as a positive value and spending as a negative value. Output would be spending totals by category, a monthly cash flow summary and a set of recommended spending reductions, together with an overall assessment of whether the user's goals are achievable.

Additionally, another input is that the user can communicate with AI and the output it will provide back is their suggesstion whether the user's budget is on tract or not and help the user what to do to achieve the goal.

3.work an example by hand,

Goal: Laptop, target $2,000, saved $500, created 26 Sep 2026, deadline 26 Mar 2027 (181 days).
- Remaining = 1500
- Months left = 181 ÷ 30 = 6 (rounded to 1 decimal)
- Needed per month = _250_
- Progress % = 25%
- Status = On track
Ask Gemini AI for the plan: Example like, "Hi, what should I do to get money in order to achieve my goal?"

4.write pseudocode

1. Settings

SET APP_CURRENCY to "AUD"
SET REQUIRED_COLUMNS to [date, description, category, amount]
SET DAYS_PER_MONTH to 30
SET CATEGORY_RULES to a dictionary: category → {cut percentage, tip}
SET DEFAULT_RULE to {cut 10%, "Review what's going into this category."}

FUNCTION load_api_key()
    TRY
        RETURN the key "GEMINI_API_KEY" from Colab Secrets
    IF that fails
        RETURN the key from the environment variables (or nothing)

SET GEMINI_API_KEY to load_api_key()
IF GEMINI_API_KEY exists
    PRINT "Gemini connected"
ELSE
    PRINT "Chat will use the offline coach"


2. Helper functions

FUNCTION text_to_date(text)
    TRY
        RETURN text converted to a date using the format YYYY-MM-DD
    IF conversion fails
        RAISE GoalError "not a valid date"

FUNCTION to_number(value)
    TRY
        RETURN value converted to a decimal number, rounded to 2 places
    IF conversion fails
        RAISE GoalError "must be a number"

FUNCTION money(amount)
    RETURN amount written as "$1,234.50"

FUNCTION savings_plan(target, current_savings, months)
    CONVERT all three inputs with to_number
    IF months <= 0
        RAISE GoalError "Months must be more than 0"
    remaining ← larger of (target − current_savings) and 0
    monthly ← remaining ÷ months
    yearly ← monthly × 12
    RETURN {remaining, monthly, fortnightly = yearly ÷ 26, weekly = yearly ÷ 52}

3. Classes

CLASS SavingsGoal
    CONSTRUCTOR(name, target, deadline, saved = 0)
        STORE name, target, saved (using to_number), created = today,
              deadline (using text_to_date), contributions = empty list
        IF name is empty           → RAISE GoalError
        IF target <= 0             → RAISE GoalError
        IF saved < 0               → RAISE GoalError
        IF deadline <= created     → RAISE GoalError
        IF saved > 0
            ADD {today, saved, "Starting balance"} to contributions

    METHOD add_contribution(amount, note)
        amount ← to_number(amount)
        IF amount = 0                → RAISE GoalError
        IF saved + amount < 0        → RAISE GoalError "can't withdraw more than saved"
        saved ← saved + amount
        ADD {today, amount, note} to contributions

    METHOD remaining()
        RETURN larger of (target − saved) and 0

    METHOD percent()
        RETURN smaller of (saved ÷ target × 100) and 100

    METHOD months_left()
        RETURN (days until deadline, minimum 0) ÷ DAYS_PER_MONTH

    METHOD monthly_needed()
        IF remaining = 0        → RETURN 0
        IF months_left < 1      → RETURN remaining      (everything is due now)
        RETURN remaining ÷ months_left

    METHOD status()
        IF remaining = 0                → RETURN "Complete"
        IF today >= deadline            → RETURN "Overdue"
        expected ← (days passed ÷ total days) × 100
        IF percent >= expected          → RETURN "On track"
        ELSE IF percent >= expected − 10 → RETURN "Slightly behind"
        ELSE                            → RETURN "Behind"

    METHOD to_dict()
        RETURN a dictionary of all the goal's info and its last 5 contributions


CLASS SavingsCoach
    CONSTRUCTOR()
        goals ← empty dictionary (lower-case name → SavingsGoal)
        transactions ← nothing

    METHOD add_goal(name, target, deadline, saved)
        goal ← new SavingsGoal(name, target, deadline, saved)
        IF lower-case name is already in goals → RAISE GoalError "already have that goal"
        STORE goal in goals
        RETURN goal

    METHOD goal_names()
        RETURN a list of every goal's name

    METHOD find_goal(name)
        IF lower-case name is not in goals → RAISE GoalError listing the goals that do exist
        RETURN that goal

    METHOD contribute(name, amount, note)
        goal ← find_goal(name)
        CALL goal.add_contribution(amount, note)
        RETURN goal

    METHOD total_monthly_needed()
        total ← 0
        FOR EACH goal in goals
            total ← total + goal.monthly_needed()
        RETURN total

    METHOD progress_table()
        FOR EACH goal, make a row with its name, saved, target, %, deadline, months left, needed, status
        RETURN the rows as a table

    METHOD require_data()
        IF transactions is nothing → RAISE GoalError "No transactions loaded yet"
        RETURN transactions

5.convert pseudocode to Python

6.test with a variety of data.